# JobHop v2 dense embedding models: BGE-M3, Qwen3-Embed, Nemotron3-Embed

This notebook evaluates the dense embedding models requested by the supervisor under the same protocol as notebook 08:

- public JobHop v2;
- ESCO occupation metadata;
- last-step next-occupation prediction;
- Markov transition score;
- dense embedding retrieval score;
- hybrid Markov + embedding alpha sweep;
- MRR / Recall@5 / Recall@10.

Models included:

- `BAAI/bge-m3`
- `Qwen/Qwen3-Embedding-0.6B`
- `nvidia/Nemotron-3-Embed-1B-BF16`

This notebook intentionally does **not** evaluate LightOn/ColBERT-style late-interaction models. Those need a separate scoring pipeline because they do not produce a single vector per text in the same way.

In [1]:
# One-click Colab setup.
!pip -q install -U pandas pyarrow requests tqdm datasets sentence-transformers accelerate transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 46.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 596.7/596.7 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 43.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.5 which is incompatible.
google-colab 1.0.0

In [2]:
import gc
import os
import random
import re
import traceback
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import requests
import torch
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

pd.set_option("display.max_columns", 140)
pd.set_option("display.max_colwidth", 180)

## Configuration

Start with `MAX_EVAL_CASES = 2000` or `5000`. These models are much heavier than MiniLM. Once a model runs successfully, increase to full evaluation by setting:

`MAX_EVAL_CASES = None`

If Colab runs out of memory, run one model at a time by commenting out the others in `MODEL_CONFIGS`.

In [3]:
DATASET_ID = "aida-ugent/JobHop"
HF_API = f"https://huggingface.co/api/datasets/{DATASET_ID}/tree/main"
SPLITS = ("train", "validation", "test")

# Safer default for larger models. Use 5000 after first success; use None for full evaluation.
MAX_EVAL_CASES = 2000
RANDOM_SEED = 7

MODEL_CONFIGS = [
    {
        "name": "BAAI/bge-m3",
        "query_prefix": "Represent this sentence for searching relevant passages: ",
        "passage_prefix": "",
        "batch_size": 32,
    },
    {
        "name": "Qwen/Qwen3-Embedding-0.6B",
        "query_prefix": "Instruct: Given a career history, retrieve the most plausible next ESCO occupation.\nQuery: ",
        "passage_prefix": "",
        "batch_size": 16,
    },
    {
        "name": "nvidia/Nemotron-3-Embed-1B-BF16",
        "query_prefix": "Instruct: Given a career history, retrieve the most plausible next ESCO occupation.\nQuery: ",
        "passage_prefix": "",
        "batch_size": 8,
    },
]

ALPHAS = [0.0, 0.1, 0.25, 0.5, 0.75, 0.9, 1.0]

REMOVE_UNKNOWN = True
COLLAPSE_CONSECUTIVE_DUPLICATES = True
RANKING_CHUNK_SIZE = 512

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEVICE

'cuda'

## Load public JobHop v2

In [4]:
def hf_tree(path=""):
    url = HF_API if not path else f"{HF_API}/{path}"
    response = requests.get(url, timeout=60)
    response.raise_for_status()
    return response.json()


def list_parquet_files():
    files = []
    stack = [""]
    while stack:
        path = stack.pop()
        try:
            entries = hf_tree(path)
        except Exception:
            continue
        for item in entries:
            item_path = item.get("path") or item.get("name")
            if not item_path:
                continue
            if item_path.endswith(".parquet"):
                files.append(item_path)
            elif item.get("type") == "directory":
                stack.append(item_path)
    return sorted(set(files))


def split_from_path(path):
    lower = path.lower()
    for split in SPLITS:
        if re.search(rf"(^|/|_|-)({split})(/|_|-|\.|$)", lower):
            return split
    if "val" in lower:
        return "validation"
    return None


parquet_files = list_parquet_files()
split_files = {split: [] for split in SPLITS}
for path in parquet_files:
    split = split_from_path(path)
    if split in split_files:
        split_files[split].append(path)

print(parquet_files)
split_files

['JobHop_v2_test.parquet', 'JobHop_v2_train.parquet', 'JobHop_v2_val.parquet']


{'train': ['JobHop_v2_train.parquet'],
 'validation': ['JobHop_v2_val.parquet'],
 'test': ['JobHop_v2_test.parquet']}

In [5]:
def hf_resolve_url(path):
    return f"https://huggingface.co/datasets/{DATASET_ID}/resolve/main/{path}"


def load_split(split):
    paths = split_files.get(split, [])
    if not paths:
        raise RuntimeError(f"No parquet files found for split={split}. Files seen: {parquet_files[:20]}")
    frames = []
    for path in tqdm(paths, desc=f"loading {split}"):
        frames.append(pd.read_parquet(hf_resolve_url(path)))
    df = pd.concat(frames, ignore_index=True)
    df["split"] = split
    return df


raw = {split: load_split(split) for split in SPLITS}
for split, df in raw.items():
    print(split, df.shape, df.columns.tolist())

loading train:   0%|          | 0/1 [00:00<?, ?it/s]

loading validation:   0%|          | 0/1 [00:00<?, ?it/s]

loading test:   0%|          | 0/1 [00:00<?, ?it/s]

train (1594827, 6) ['resume_id', 'matched_code', 'start_date', 'end_date', 'university_level', 'split']
validation (199587, 6) ['resume_id', 'matched_code', 'start_date', 'end_date', 'university_level', 'split']
test (198877, 6) ['resume_id', 'matched_code', 'start_date', 'end_date', 'university_level', 'split']


In [6]:
def standardize_jobhop_schema(df):
    df = df.copy()
    if "person_id" not in df.columns:
        if "resume_id" in df.columns:
            df["person_id"] = df["resume_id"]
        else:
            raise KeyError(f"No person/resume identifier column found. Columns: {df.columns.tolist()}")

    if "university_studies" not in df.columns:
        if "university_level" in df.columns:
            df["university_studies"] = df["university_level"]
        else:
            df["university_studies"] = pd.NA

    df["matched_code"] = df["matched_code"].astype(str).str.strip()
    return df


raw = {split: standardize_jobhop_schema(df) for split, df in raw.items()}
pd.DataFrame([
    {
        "split": split,
        "rows": len(df),
        "resumes": df["person_id"].nunique(),
        "unknown_code_rows": (df["matched_code"].str.lower() == "unknown").sum(),
        "missing_start_date": df["start_date"].isna().sum(),
        "missing_end_date": df["end_date"].isna().sum(),
    }
    for split, df in raw.items()
])

,split,rows,resumes,unknown_code_rows,missing_start_date,missing_end_date
0,train,1594827,284251,110573,58312,136497
1,validation,199587,35531,13622,7329,17041
2,test,198877,35533,13587,7085,16829


## Load ESCO metadata

In [7]:
def clean_text(value):
    if value is None or pd.isna(value):
        return ""
    return str(value).replace("\n", " ").strip()


def normalize_esco_code(value):
    if value is None or pd.isna(value):
        return ""
    return str(value).strip().rstrip("/").split("/")[-1]


def add_meta(lookup, code, label="", description="", source=""):
    code = normalize_esco_code(code)
    if not code:
        return
    label = clean_text(label)
    description = clean_text(description)
    if code not in lookup or (not lookup[code].get("description") and description):
        lookup[code] = {
            "label": label or lookup.get(code, {}).get("label", ""),
            "description": description or lookup.get(code, {}).get("description", ""),
            "source": source,
        }


def load_danieldux_esco_metadata():
    lookup = {}
    frames = []
    parquet_urls = [
        "https://huggingface.co/datasets/danieldux/ESCO/resolve/main/data/en/train.parquet",
        "https://huggingface.co/datasets/danieldux/ESCO/resolve/main/data/en/test.parquet",
    ]
    for url in parquet_urls:
        try:
            frames.append(pd.read_parquet(url))
            print("Loaded danieldux parquet:", url)
        except Exception as exc:
            print("Could not read danieldux parquet:", url, exc)

    if not frames:
        try:
            ds = load_dataset("danieldux/ESCO", split="train")
            frames.append(ds.to_pandas())
            print("Loaded danieldux/ESCO through datasets")
        except Exception as exc:
            print("Could not load danieldux/ESCO through datasets:", exc)

    if frames:
        df = pd.concat(frames, ignore_index=True).drop_duplicates(subset=["ESCO_CODE"])
        print("Loaded danieldux/ESCO:", df.shape, df.columns.tolist())
        for _, row in df.iterrows():
            add_meta(
                lookup,
                row.get("ESCO_CODE"),
                row.get("ESCO_OCCUPATION", ""),
                row.get("ESCO_DESCRIPTION", ""),
                "hf_danieldux_esco",
            )
    print("danieldux lookup entries:", len(lookup))
    return lookup


metadata_lookup = load_danieldux_esco_metadata()
list(metadata_lookup.items())[:3]

Loaded danieldux parquet: https://huggingface.co/datasets/danieldux/ESCO/resolve/main/data/en/train.parquet
Loaded danieldux parquet: https://huggingface.co/datasets/danieldux/ESCO/resolve/main/data/en/test.parquet
Loaded danieldux/ESCO: (3008, 19) ['ISCO_CODE_1', 'ISCO_LABEL_1', 'ISCO_DEFINITION_1', 'ISCO_CODE_2', 'ISCO_LABEL_2', 'ISCO_DEFINITION_2', 'ISCO_CODE_3', 'ISCO_LABEL_3', 'ISCO_DEFINITION_3', 'ISCO_CODE_4', 'ISCO_LABEL_4', 'ISCO_DEFINITION_4', 'ISCO_CODES', 'ISCO_LABELS', 'ESCO_CODE', 'ESCO_LABELS', 'ESCO_OCCUPATION', 'ESCO_DESCRIPTION', 'LANGUAGE']
danieldux lookup entries: 3007


[('1223.1',
  {'label': 'product manager',
   'description': 'Product managers are responsible for managing the lifecycle of a product. They research and develop new products in addition to managing existing ones through market research and strategic planning. Product managers perform marketing and planning activities to increase profits.',
   'source': 'hf_danieldux_esco'}),
 ('3324.4.8',
  {'label': 'wholesale merchant in computers, computer peripheral equipment and software',
   'description': 'Wholesale merchants in computers, computer peripheral equipment and software investigate potential wholesale buyers and suppliers and match their needs. They conclude trades involving large quantities of goods.',
   'source': 'hf_danieldux_esco'}),
 ('7543.9.3',
  {'label': 'metal product quality control inspector',
   'description': 'Metal product quality control inspectors perform preventive and operational quality control on the metal products. They examine the materials at various stages 

## Build trajectories and cases

In [8]:
def date_key(value):
    if pd.isna(value):
        return (9999, 9)
    text = str(value).strip()
    if not text or text in {"-", "nan", "None"}:
        return (9999, 9)
    m = re.search(r"(19|20)\d{2}", text)
    year = int(m.group(0)) if m else 9999
    q = 9
    qm = re.search(r"[Qq][-_ ]?([1-4])", text)
    if qm:
        q = int(qm.group(1))
    return (year, q)


def normalize_jobhop_code(value):
    if value is None or pd.isna(value):
        return None
    text = str(value).strip()
    if not text or text.lower() in {"nan", "none"}:
        return None
    if REMOVE_UNKNOWN and text.lower() == "unknown":
        return None
    return text


def compact_consecutive(seq):
    out = []
    for item in seq:
        if not out or item["code"] != out[-1]["code"]:
            out.append(item)
    return out


def make_trajectories(df):
    trajectories = {}
    for person_id, group in tqdm(df.groupby("person_id", sort=False), desc="trajectories"):
        group = group.copy()
        group["_date_key"] = group["start_date"].apply(date_key)
        group = group.sort_values(["_date_key", "end_date", "matched_code"], kind="mergesort")
        seq = []
        for _, row in group.iterrows():
            code = normalize_jobhop_code(row["matched_code"])
            if code is None:
                continue
            seq.append({
                "code": code,
                "education": row.get("university_studies"),
            })
        if COLLAPSE_CONSECUTIVE_DUPLICATES:
            seq = compact_consecutive(seq)
        if len(seq) >= 2:
            trajectories[person_id] = seq
    return trajectories


trajectories = {split: make_trajectories(raw[split]) for split in SPLITS}
pd.DataFrame([
    {"split": split, "trajectories_len_ge_2": len(traj)}
    for split, traj in trajectories.items()
])

trajectories:   0%|          | 0/284251 [00:00<?, ?it/s]

trajectories:   0%|          | 0/35531 [00:00<?, ?it/s]

trajectories:   0%|          | 0/35533 [00:00<?, ?it/s]

,split,trajectories_len_ge_2
0,train,256593
1,validation,32123
2,test,32089


In [9]:
def make_last_step_cases(trajectories_dict):
    cases = []
    for person_id, seq in trajectories_dict.items():
        cases.append({
            "person_id": person_id,
            "prefix": [item["code"] for item in seq[:-1]],
            "target": seq[-1]["code"],
            "education": seq[0].get("education"),
            "prefix_length": len(seq) - 1,
        })
    return cases


validation_cases = make_last_step_cases(trajectories["validation"])
test_cases = make_last_step_cases(trajectories["test"])

candidate_codes = sorted({item["code"] for seq in trajectories["train"].values() for item in seq})
candidate_set = set(candidate_codes)
code_to_index = {code: i for i, code in enumerate(candidate_codes)}

validation_cases = [case for case in validation_cases if case["target"] in candidate_set]
test_cases = [case for case in test_cases if case["target"] in candidate_set]

if MAX_EVAL_CASES is not None:
    rng = random.Random(RANDOM_SEED)
    validation_cases = rng.sample(validation_cases, min(MAX_EVAL_CASES, len(validation_cases)))
    test_cases = rng.sample(test_cases, min(MAX_EVAL_CASES, len(test_cases)))

metadata_coverage = sum(1 for c in candidate_codes if c in metadata_lookup)
description_coverage = sum(1 for c in candidate_codes if metadata_lookup.get(c, {}).get("description"))

print("candidate occupations:", len(candidate_codes))
print("candidate metadata coverage:", metadata_coverage, f"({metadata_coverage / len(candidate_codes):.1%})")
print("candidate description coverage:", description_coverage, f"({description_coverage / len(candidate_codes):.1%})")
print("validation eval cases:", len(validation_cases))
print("test eval cases:", len(test_cases))

candidate occupations: 2982
candidate metadata coverage: 2982 (100.0%)
candidate description coverage: 2982 (100.0%)
validation eval cases: 2000
test eval cases: 2000


## Build candidate and query texts

In [10]:
def label_for_code(code):
    meta = metadata_lookup.get(code, {})
    return meta.get("label") or f"ESCO code {code}"


def description_for_code(code):
    return metadata_lookup.get(code, {}).get("description", "")


def compact_desc(text, max_chars=500):
    text = clean_text(text)
    if len(text) > max_chars:
        return text[:max_chars].rsplit(" ", 1)[0] + "..."
    return text


def candidate_text(code):
    label = label_for_code(code)
    desc = compact_desc(description_for_code(code))
    pieces = [f"ESCO occupation code: {code}.", f"Occupation title: {label}."]
    if desc:
        pieces.append(f"Description: {desc}.")
    return " ".join(pieces)


def query_text(case):
    labels = [label_for_code(code) for code in case["prefix"]]
    history = " -> ".join(labels)
    codes = " -> ".join(case["prefix"])
    edu = case.get("education")
    edu_text = "" if pd.isna(edu) else f" Education level: {edu}."
    return (
        f"Career history occupations: {history}. "
        f"Career history ESCO codes: {codes}."
        f"{edu_text} Predict the next ESCO occupation."
    )


candidate_texts = [candidate_text(code) for code in candidate_codes]
validation_query_texts = [query_text(case) for case in validation_cases]
test_query_texts = [query_text(case) for case in test_cases]

print(candidate_texts[0])
print(test_query_texts[0])

ESCO occupation code: 0110.1. Occupation title: air force officer. Description: Air force officers specialise in flying or ground duties, and supervise a team of air force personnel. They coordinate their team's training and welfare, and perform duties specific to their area of specialisation..
Career history occupations: waiter/waitress. Career history ESCO codes: 5131.2. Education level: Secondary school. Predict the next ESCO occupation.


## Markov scores

In [11]:
def train_markov(trajectories_dict):
    transitions = defaultdict(Counter)
    global_next = Counter()
    for seq in trajectories_dict.values():
        codes = [item["code"] for item in seq]
        for a, b in zip(codes[:-1], codes[1:]):
            transitions[a][b] += 1
            global_next[b] += 1

    global_total = sum(global_next.values())
    global_probs = np.zeros(len(candidate_codes), dtype="float32")
    for code, count in global_next.items():
        if code in code_to_index:
            global_probs[code_to_index[code]] = count / global_total

    transition_probs = {}
    for prev, counter in transitions.items():
        total = sum(counter.values())
        indices = []
        probs = []
        for nxt, count in counter.items():
            if nxt in code_to_index:
                indices.append(code_to_index[nxt])
                probs.append(count / total)
        transition_probs[prev] = (np.array(indices, dtype=np.int32), np.array(probs, dtype="float32"))
    return transition_probs, global_probs


transition_probs, global_probs = train_markov(trajectories["train"])


def markov_score_matrix(cases):
    scores = np.zeros((len(cases), len(candidate_codes)), dtype="float32")
    for i, case in enumerate(tqdm(cases, desc="markov score rows")):
        last = case["prefix"][-1]
        if last in transition_probs:
            idx, probs = transition_probs[last]
            scores[i, idx] = probs
        else:
            scores[i, :] = global_probs
    return scores


validation_markov_scores = markov_score_matrix(validation_cases)
test_markov_scores = markov_score_matrix(test_cases)

markov score rows:   0%|          | 0/2000 [00:00<?, ?it/s]

markov score rows:   0%|          | 0/2000 [00:00<?, ?it/s]

## Dense embedding and hybrid evaluation helpers

In [12]:
def apply_prefixes(texts, prefix):
    return [prefix + text for text in texts] if prefix else texts


def load_sentence_transformer_model(model_name):
    try:
        return SentenceTransformer(model_name, device=DEVICE, trust_remote_code=True)
    except TypeError:
        return SentenceTransformer(model_name, device=DEVICE, model_kwargs={"trust_remote_code": True})


def encode_texts(model, texts, prefix, batch_size):
    formatted = apply_prefixes(texts, prefix)
    emb = model.encode(
        formatted,
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )
    return emb.astype("float32")


def row_minmax(x):
    mn = x.min(axis=1, keepdims=True)
    mx = x.max(axis=1, keepdims=True)
    denom = np.maximum(mx - mn, 1e-8)
    return (x - mn) / denom


def row_maxnorm(x):
    mx = x.max(axis=1, keepdims=True)
    return x / np.maximum(mx, 1e-8)


validation_markov_norm = row_maxnorm(validation_markov_scores)
test_markov_norm = row_maxnorm(test_markov_scores)


def evaluate_score_matrix(score_matrix, cases):
    target_indices = np.array([code_to_index[case["target"]] for case in cases], dtype=np.int64)
    rr, r5, r10 = [], [], []
    for start in range(0, len(cases), RANKING_CHUNK_SIZE):
        end = min(start + RANKING_CHUNK_SIZE, len(cases))
        chunk = score_matrix[start:end]
        kth = min(10, chunk.shape[1] - 1)
        order = np.argpartition(-chunk, kth=kth, axis=1)[:, :10]
        top_scores = np.take_along_axis(chunk, order, axis=1)
        sorted_local = np.argsort(-top_scores, axis=1)
        top10 = np.take_along_axis(order, sorted_local, axis=1)
        for row_idx, preds in enumerate(top10):
            target = target_indices[start + row_idx]
            matches = np.where(preds == target)[0]
            rr.append(1.0 / (int(matches[0]) + 1) if len(matches) else 0.0)
            r5.append(float(target in preds[:5]))
            r10.append(float(target in preds[:10]))
    return {
        "MRR": float(np.mean(rr)),
        "Recall@5": float(np.mean(r5)),
        "Recall@10": float(np.mean(r10)),
        "n_cases": len(cases),
    }

## Run models

In [13]:
all_results = []
failures = []

for cfg in MODEL_CONFIGS:
    model_name = cfg["name"]
    print("\n" + "=" * 120)
    print("Model:", model_name)
    try:
        model = load_sentence_transformer_model(model_name)
        candidate_embeddings = encode_texts(model, candidate_texts, cfg.get("passage_prefix", ""), cfg.get("batch_size", 16))
        validation_embeddings = encode_texts(model, validation_query_texts, cfg.get("query_prefix", ""), cfg.get("batch_size", 16))
        test_embeddings = encode_texts(model, test_query_texts, cfg.get("query_prefix", ""), cfg.get("batch_size", 16))

        validation_embedding_norm = row_minmax(validation_embeddings @ candidate_embeddings.T)
        test_embedding_norm = row_minmax(test_embeddings @ candidate_embeddings.T)

        for split, cases, emb_norm, markov_norm in [
            ("validation", validation_cases, validation_embedding_norm, validation_markov_norm),
            ("test", test_cases, test_embedding_norm, test_markov_norm),
        ]:
            for alpha in ALPHAS:
                final_scores = alpha * markov_norm + (1.0 - alpha) * emb_norm
                scores = evaluate_score_matrix(final_scores, cases)
                row = {
                    "model": model_name,
                    "split": split,
                    "alpha_markov": alpha,
                    "alpha_embedding": 1.0 - alpha,
                    "max_eval_cases": MAX_EVAL_CASES,
                    **scores,
                }
                all_results.append(row)
                print(row)

        del model, candidate_embeddings, validation_embeddings, test_embeddings
        del validation_embedding_norm, test_embedding_norm
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    except Exception as exc:
        error_text = "".join(traceback.format_exception_only(type(exc), exc)).strip()
        print("FAILED:", model_name, error_text)
        failures.append({
            "model": model_name,
            "error": error_text,
        })
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

results_df = pd.DataFrame(all_results)
failures_df = pd.DataFrame(failures)

display(results_df.sort_values(["model", "split", "alpha_markov"]).style.format({
    "MRR": "{:.4f}",
    "Recall@5": "{:.4f}",
    "Recall@10": "{:.4f}",
}))
if len(failures_df):
    display(failures_df)

results_df.to_csv("jobhop_v2_dense_new_models_hybrid_results.csv", index=False)
failures_df.to_csv("jobhop_v2_dense_new_models_failures.csv", index=False)
print("Saved result CSVs.")


Model: BAAI/bge-m3


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

... (49 outputs left out)

Batches:   0%|          | 0/250 [00:00<?, ?it/s]

{'model': 'nvidia/Nemotron-3-Embed-1B-BF16', 'split': 'validation', 'alpha_markov': 0.0, 'alpha_embedding': 1.0, 'max_eval_cases': 2000, 'MRR': 0.034941269841269836, 'Recall@5': 0.0525, 'Recall@10': 0.0655, 'n_cases': 2000}
{'model': 'nvidia/Nemotron-3-Embed-1B-BF16', 'split': 'validation', 'alpha_markov': 0.1, 'alpha_embedding': 0.9, 'max_eval_cases': 2000, 'MRR': 0.04896269841269841, 'Recall@5': 0.077, 'Recall@10': 0.0905, 'n_cases': 2000}
{'model': 'nvidia/Nemotron-3-Embed-1B-BF16', 'split': 'validation', 'alpha_markov': 0.25, 'alpha_embedding': 0.75, 'max_eval_cases': 2000, 'MRR': 0.08816507936507935, 'Recall@5': 0.129, 'Recall@10': 0.1555, 'n_cases': 2000}
{'model': 'nvidia/Nemotron-3-Embed-1B-BF16', 'split': 'validation', 'alpha_markov': 0.5, 'alpha_embedding': 0.5, 'max_eval_cases': 2000, 'MRR': 0.12365892857142856, 'Recall@5': 0.193, 'Recall@10': 0.2565, 'n_cases': 2000}
{'model': 'nvidia/Nemotron-3-Embed-1B-BF16', 'split': 'validation', 'alpha_markov': 0.75, 'alpha_embedding':

,model,split,alpha_markov,alpha_embedding,max_eval_cases,MRR,Recall@5,Recall@10,n_cases
7,BAAI/bge-m3,test,0.000000,1.000000,2000,0.0650,0.1145,0.1535,2000
8,BAAI/bge-m3,test,0.100000,0.900000,2000,0.0882,0.1465,0.1970,2000
9,BAAI/bge-m3,test,0.250000,0.750000,2000,0.1304,0.2025,0.2695,2000
10,BAAI/bge-m3,test,0.500000,0.500000,2000,0.1485,0.2320,0.3155,2000
11,BAAI/bge-m3,test,0.750000,0.250000,2000,0.1446,0.2330,0.3135,2000
12,BAAI/bge-m3,test,0.900000,0.100000,2000,0.1423,0.2300,0.3125,2000
13,BAAI/bge-m3,test,1.000000,0.000000,2000,0.1391,0.2230,0.3085,2000
0,BAAI/bge-m3,validation,0.000000,1.000000,2000,0.0669,0.1180,0.1520,2000
1,BAAI/bge-m3,validation,0.100000,0.900000,2000,0.0911,0.1445,0.1945,2000
2,BAAI/bge-m3,validation,0.250000,0.750000,2000,0.1259,0.1990,0.2555,2000


Saved result CSVs.


In [14]:
if len(results_df):
    best = results_df.sort_values(["split", "MRR"], ascending=[True, False]).groupby("split").head(10)
    reference_rows = [
        {
            "model": "MiniLM + Markov hybrid, previous 08 full test",
            "split": "test",
            "alpha_markov": 0.50,
            "MRR": 0.1343,
            "Recall@5": 0.2153,
            "Recall@10": 0.2926,
            "note": "Full test result from notebook 08.",
        },
        {
            "model": "Pure Markov, previous 08 full test",
            "split": "test",
            "alpha_markov": 1.00,
            "MRR": 0.1312,
            "Recall@5": 0.2069,
            "Recall@10": 0.2919,
            "note": "Full test transition-count baseline.",
        },
        {
            "model": "ROUTE+STEP, reported in STEP paper",
            "split": "test",
            "MRR": 0.2220,
            "Recall@5": 0.3130,
            "Recall@10": 0.4040,
            "note": "Contextual only; not directly comparable until exact preprocessing is aligned.",
        },
    ]
    comparison_df = pd.concat([best, pd.DataFrame(reference_rows)], ignore_index=True, sort=False)
    display(comparison_df[["model", "split", "alpha_markov", "MRR", "Recall@5", "Recall@10", "note"]].style.format({
        "alpha_markov": "{:.2f}",
        "MRR": "{:.4f}",
        "Recall@5": "{:.4f}",
        "Recall@10": "{:.4f}",
    }))

,model,split,alpha_markov,MRR,Recall@5,Recall@10,note
0,BAAI/bge-m3,test,0.50,0.1485,0.2320,0.3155,nan
1,Qwen/Qwen3-Embedding-0.6B,test,0.50,0.1479,0.2310,0.3200,nan
2,BAAI/bge-m3,test,0.75,0.1446,0.2330,0.3135,nan
3,Qwen/Qwen3-Embedding-0.6B,test,0.75,0.1432,0.2300,0.3160,nan
4,BAAI/bge-m3,test,0.90,0.1423,0.2300,0.3125,nan
5,Qwen/Qwen3-Embedding-0.6B,test,0.90,0.1414,0.2275,0.3120,nan
6,nvidia/Nemotron-3-Embed-1B-BF16,test,0.90,0.1397,0.2270,0.3095,nan
7,nvidia/Nemotron-3-Embed-1B-BF16,test,0.75,0.1394,0.2260,0.3050,nan
8,BAAI/bge-m3,test,1.00,0.1391,0.2230,0.3085,nan
9,Qwen/Qwen3-Embedding-0.6B,test,1.00,0.1391,0.2230,0.3085,nan


## How to report this notebook

Report this as a screening evaluation of the dense new models requested by the supervisor.

Be careful with compute failures:

- If BGE-M3 and Qwen3 run but Nemotron fails, report the failure as a compute/access issue, not as a model result.
- If all models run only on `MAX_EVAL_CASES = 2000`, describe the numbers as sampled preliminary results.
- Only compare directly with notebook 08 full-test numbers after setting `MAX_EVAL_CASES = None`.